# 13.1 视觉语言 (Vision-Language)

视觉语言模型将视觉信息与语言理解结合，实现图像理解和视觉问答。

本节涵盖：视觉编码器、视觉语言模型、视觉指令微调

## 1. 视觉编码器与视觉语言模型

**视觉编码器**：将图像转换为向量表示。
- **ViT**：将图像切分为patch，用Transformer编码
- **CLIP**：对比学习训练的视觉-语言双编码器

**视觉语言模型（VLM）**：将视觉特征投影到语言空间，与文本token一起输入LLM。
- **LLaVA**：ViT + MLP投影 + LLaMA
- **Qwen-VL**：ViT + 交叉注意力 + Qwen

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

torch.manual_seed(42)

class SimpleViT(nn.Module):
    def __init__(self, patch_size=4, d_model=64, n_heads=2, n_layers=1, img_size=16):
        super().__init__()
        n_patches = (img_size // patch_size) ** 2
        self.patch_embed = nn.Linear(3 * patch_size * patch_size, d_model)
        self.pos_embed = nn.Parameter(torch.randn(1, n_patches, d_model) * 0.02)
        layer = nn.TransformerEncoderLayer(d_model, n_heads, d_model * 4, batch_first=True)
        self.encoder = nn.TransformerEncoder(layer, n_layers)

    def forward(self, img):
        B, C, H, W = img.shape
        ps = int(math.sqrt(img.shape[2] * img.shape[3] // (B * 3 * 64)))
        patches = img.reshape(B, C, H // 4, 4, W // 4, 4).permute(0, 2, 4, 1, 3, 5).reshape(B, -1, C * 16)
        h = self.patch_embed(patches) + self.pos_embed
        return self.encoder(h)

class SimpleVLM(nn.Module):
    def __init__(self, d_vision=64, d_language=128, vocab_size=500):
        super().__init__()
        self.vision_encoder = SimpleViT()
        self.vision_proj = nn.Linear(d_vision, d_language)
        self.text_embed = nn.Embedding(vocab_size, d_language)
        self.lm = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_language, 2, d_language * 4, batch_first=True), 2
        )
        self.head = nn.Linear(d_language, vocab_size)

    def forward(self, image, text_tokens):
        vis_features = self.vision_encoder(image)
        vis_tokens = self.vision_proj(vis_features)
        txt_tokens = self.text_embed(text_tokens)
        combined = torch.cat([vis_tokens, txt_tokens], dim=1)
        h = self.lm(combined)
        return self.head(h)

vit = SimpleViT()
img = torch.randn(2, 3, 16, 16)
vis_out = vit(img)

vlm = SimpleVLM()
text = torch.randint(0, 500, (2, 10))
logits = vlm(img, text)

print('=== Vision-Language Model ===')
print(f'Image: {img.shape} -> Vision features: {vis_out.shape}')
print(f'Text tokens: {text.shape}')
print(f'Combined -> VLM output: {logits.shape}')
print(f'\nKey: Vision tokens + Text tokens -> unified LLM processing.')
print(f'Vision projection aligns visual features with language space.')

## 2. 视觉指令微调

**目的**：使VLM能遵循视觉相关指令

**基本原理**：收集图像-指令-回复三元组数据，微调VLM使其能根据图像回答问题、描述内容等。

**LLaVA训练流程**：
1. 预训练：对齐视觉投影层（冻结LLM）
2. 指令微调：解冻LLM，在视觉指令数据上微调

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

print('=== Visual Instruction Tuning ===')
print(f'\nTraining data format:')
print(f'  Image: [visual input]')
print(f'  Instruction: "What is in this image?"')
print(f'  Response: "The image shows a cat sitting on a sofa."')
print(f'\nTwo-stage training:')
print(f'  Stage 1: Pretrain projection (freeze LLM, align vision-language)')
print(f'  Stage 2: Instruction fine-tune (unfreeze LLM, learn to follow instructions)')
print(f'\nKey: Visual instruction tuning teaches VLMs to follow visual commands,')
print(f'enabling tasks like VQA, image captioning, and visual reasoning.')